In [1]:
# mujoco: 物理シミュレータ / gymnasium: 強化学習環境のひな形
# stable-baselines3: PPO（強化学習） / imageio: 動画保存
!pip install mujoco gymnasium stable-baselines3 imageio imageio-ffmpeg
# 古いgymを削除（警告を消すため。入っていなくてもOK）
!pip uninstall -y gym

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 18.3 MB/s eta 0:00:00
Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2


In [2]:
# ============================================================
# 準備：ライブラリの読み込み
#   ※MuJoCoの描画機能(OpenGL)は使わない → 描画エラーが起きない
# ============================================================
import os
import numpy as np            # 数値計算
import mujoco                 # 物理シミュレータ本体（物理計算だけに使う）
import gymnasium as gym       # 強化学習環境のひな形
from gymnasium import spaces  # 行動・観測の範囲や形を定義する道具

# 動画の絵はmatplotlibで描く（画面のない環境でも動く設定）
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# ロボットの設計図（MJCF形式のXML）
#   構造：土台 → 関節1(j1) → リンク1 → 関節2(j2) → リンク2 → 手先(tip)
# ============================================================
XML = """
<mujoco model="arm2dof">
  <!-- 物理計算の刻み幅0.01秒、重力0（真上から見た平面の動きにするため） -->
  <option timestep="0.01" gravity="0 0 0"/>

  <worldbody>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>

    <!-- 1本目の腕：z軸まわりに回る関節1。damping＝回転の抵抗 -->
    <body name="link1" pos="0 0 0.05">
      <joint name="j1" type="hinge" axis="0 0 1" damping="0.5"/>
      <geom type="capsule" fromto="0 0 0 0.2 0 0" size="0.02"/>

      <!-- 2本目の腕：リンク1の先端に付く。関節2は±150度まで -->
      <body name="link2" pos="0.2 0 0">
        <joint name="j2" type="hinge" axis="0 0 1" limited="true" range="-150 150" damping="0.5"/>
        <geom type="capsule" fromto="0 0 0 0.2 0 0" size="0.018"/>
        <site name="tip" pos="0.2 0 0" size="0.02"/>   <!-- 手先の目印 -->
      </body>
    </body>

    <!-- 目標（緑の球）：プログラムから位置を動かせる。当たり判定なし -->
    <body name="target" mocap="true" pos="0.2 0.2 0.05">
      <geom type="sphere" size="0.025" contype="0" conaffinity="0"/>
    </body>
  </worldbody>

  <!-- モータ：各関節に1つずつ。指令値は-1〜+1 -->
  <actuator>
    <motor joint="j1" ctrllimited="true" ctrlrange="-1 1"/>
    <motor joint="j2" ctrllimited="true" ctrlrange="-1 1"/>
  </actuator>
</mujoco>
"""

# ============================================================
# 強化学習の「環境」クラス（__init__ / reset / step / render）
# ============================================================
class Arm2Env(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 50}

    def __init__(self, render_mode=None):
        self.model = mujoco.MjModel.from_xml_string(XML)   # XMLからロボットを作る
        self.data = mujoco.MjData(self.model)               # 今の状態の入れ物
        # 行動：2つのモータへの指令値（各-1〜+1）
        self.action_space = spaces.Box(-1, 1, (2,), np.float32)
        # 観測：10個の数値（内訳は_obs）
        self.observation_space = spaces.Box(-np.inf, np.inf, (10,), np.float64)
        self.render_mode = render_mode
        self.fig = None      # 描画用の図（使うときに作る）
        self.t = 0           # ステップ数

    def _obs(self):
        """AIに渡す観測（10個）"""
        q = self.data.qpos                      # 関節角度
        tip = self.data.site("tip").xpos[:2]    # 手先の位置(x, y)
        tgt = self.data.mocap_pos[0][:2]        # 目標の位置(x, y)
        return np.concatenate([
            np.cos(q), np.sin(q),   # 角度をcos・sinで表す（4個）
            self.data.qvel,         # 関節の回転速度（2個）
            tgt,                    # 目標位置（2個）
            tip - tgt,              # 手先と目標のズレ（2個）
        ])

    def reset(self, seed=None, options=None):
        """1回の挑戦の開始：アームと目標を初期配置する"""
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = self.np_random.uniform(-0.1, 0.1, 2)   # 初期角度を少しランダムに
        r = self.np_random.uniform(0.1, 0.38)                      # 目標までの距離（届く範囲）
        th = self.np_random.uniform(-np.pi, np.pi)                 # 目標の方向
        self.data.mocap_pos[0] = [r*np.cos(th), r*np.sin(th), 0.05]
        mujoco.mj_forward(self.model, self.data)                   # 配置を反映
        self.t = 0
        return self._obs(), {}

    def step(self, a):
        """AIの行動を受け取り、0.02秒進めて結果を返す"""
        self.data.ctrl[:] = a
        for _ in range(2):
            mujoco.mj_step(self.model, self.data)
        self.t += 1
        obs = self._obs()
        dist = np.linalg.norm(obs[-2:])                  # 手先と目標の距離
        # 報酬：目標に近いほど高く、大きな力を使うと少し減点
        reward = -dist - 0.01*np.sum(np.square(a))
        # (観測, 報酬, 途中終了, 時間切れ(200ステップ), 追加情報)
        return obs, reward, False, self.t >= 200, {"dist": dist}

    def render(self):
        """matplotlibで真上から見た図を描き、画像(480×480)として返す"""
        if self.fig is None:                              # 初回だけ図を作る
            self.fig, self.ax = plt.subplots(figsize=(4.8, 4.8), dpi=100)
        ax = self.ax
        ax.clear()                                        # 前のコマを消す
        ax.set_xlim(-0.45, 0.45); ax.set_ylim(-0.45, 0.45)
        ax.set_aspect("equal"); ax.grid(alpha=0.3)

        # MuJoCoが計算した位置を取り出す
        elbow = self.data.body("link2").xpos[:2]          # ひじ（関節2）の位置
        tip   = self.data.site("tip").xpos[:2]            # 手先の位置
        tgt   = self.data.mocap_pos[0][:2]                # 目標の位置

        # 腕を線で描く（青＝リンク1、オレンジ＝リンク2）
        ax.plot([0, elbow[0]], [0, elbow[1]], lw=8, color=(.2, .4, .8), solid_capstyle="round")
        ax.plot([elbow[0], tip[0]], [elbow[1], tip[1]], lw=7, color=(.8, .4, .2), solid_capstyle="round")
        ax.plot(0, 0, "ko", ms=8)                                        # 土台（関節1）
        ax.add_patch(plt.Circle(tgt, 0.025, color="lime", alpha=0.6))    # 目標（緑の球）

        # 経過時間と、手先と目標の距離を表示
        ax.set_title(f"t = {self.t*0.02:.2f} s   dist = {np.linalg.norm(tip - tgt):.3f} m")

        self.fig.canvas.draw()                            # 図を画像に変換
        return np.asarray(self.fig.canvas.buffer_rgba())[:, :, :3].copy()

print("環境の定義OK")

環境の定義OK


In [3]:
env_test = Arm2Env(render_mode="rgb_array")
env_test.reset(seed=0)
print("描画OK:", env_test.render().shape)   # (480, 480, 3) と出ればOK

描画OK: (480, 480, 3)


In [4]:
# ここで落ちなければ、描画の設定は問題なし
env_test = Arm2Env(render_mode="rgb_array")
env_test.reset(seed=0)
img = env_test.render()
print("描画OK:", img.shape)   # (480, 480, 3) と出ればOK

描画OK: (480, 480, 3)


In [5]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# 環境を8個同時に動かして、学習を速くする
env = make_vec_env(Arm2Env, n_envs=8)

# PPOモデル（MLP方策）。tensorboard_logは学習曲線をレポートに使うため
model = PPO("MlpPolicy", env, verbose=1, n_steps=1024, batch_size=256,
            tensorboard_log="logs")

# 200万ステップ学習（CPUで十数分〜20分程度）
#   ep_rew_mean が0に近づけば上達している
model.learn(total_timesteps=2000_000)

model.save("ppo_arm2")   # ppo_arm2.zip として保存

# ★Googleドライブにも保存（カーネルが落ちても消えないように）
from google.colab import drive
drive.mount('/content/drive')
model.save("/content/drive/MyDrive/ppo_arm2")

Using cpu device
Logging to logs/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -94      |
| time/              |          |
|    fps             | 6273     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 8192     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 200          |
|    ep_rew_mean          | -90.2        |
| time/                   |              |
|    fps                  | 4177         |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 16384        |
| train/                  |              |
|    approx_kl            | 0.0030601136 |
|    clip_fraction        | 0.0142       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.82        |
|    explained_vari

In [6]:
import imageio
from stable_baselines3 import PPO

env = Arm2Env(render_mode="rgb_array")
# ★Colabにファイルがなければドライブから読み込む
path = "ppo_arm2" if os.path.exists("ppo_arm2.zip") else "/content/drive/MyDrive/ppo_arm2"
model = PPO.load(path)

frames = []
for ep in range(20):                     # ★5 → 20回（約80秒の動画。課題の1分以上を満たす）
    obs, _ = env.reset(seed=ep)          # seedを変えると目標の位置が変わる
    for _ in range(200):                 # 200ステップ＝約4秒
        action, _ = model.predict(obs, deterministic=True)   # AIが行動を決める
        obs, r, term, trunc, info = env.step(action)
        frames.append(env.render())
    print(f"ep{ep} 最終距離: {info['dist']:.3f} m")         # 0.02m以下なら成功の目安

imageio.mimsave("arm2_result.mp4", frames, fps=50)
print("動画を保存しました")

ep0 最終距離: 0.049 m
ep1 最終距離: 0.046 m
ep2 最終距離: 0.227 m
ep3 最終距離: 0.069 m
ep4 最終距離: 0.272 m
ep5 最終距離: 0.141 m
ep6 最終距離: 0.049 m
ep7 最終距離: 0.220 m
ep8 最終距離: 0.025 m
ep9 最終距離: 0.037 m
ep10 最終距離: 0.232 m
ep11 最終距離: 0.168 m
ep12 最終距離: 0.051 m
ep13 最終距離: 0.229 m
ep14 最終距離: 0.084 m
ep15 最終距離: 0.098 m
ep16 最終距離: 0.030 m
ep17 最終距離: 0.106 m
ep18 最終距離: 0.080 m
ep19 最終距離: 0.079 m
動画を保存しました


In [7]:
from IPython.display import Video
Video("arm2_result.mp4", embed=True, width=480)   # Colab上で再生

In [8]:
from google.colab import files
files.download("arm2_result.mp4")                 # PCにダウンロード

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>